In [ ]:
import warnings
warnings.filterwarnings('ignore')

import itertools
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.stats.diagnostic import acorr_ljungbox

#CONFIG
CSV_PATH    = "Dengue_Climate_Bangladesh.csv"
OUTPUT_PNG  = "arimax_prediction.png"
DIAG_PNG    = "arimax_residual_diagnostics.png"
RESULTS_CSV = "arimax_2025_predictions.csv"

# 1. DATA LOADING & PREPROCESSING
df = pd.read_csv(CSV_PATH)
df['DATE'] = pd.to_datetime(df['YEAR'].astype(str) + '-' + df['MONTH'].astype(str) + '-01')
df = df.sort_values('DATE').reset_index(drop=True)
df.set_index('DATE', inplace=True)
df.index.freq = 'MS'

# 2. FEATURE ENGINEERING (collinearity-screened, VIF < 12 — see chat)
df['HUMIDITY_lag_1'] = df['HUMIDITY'].shift(1)
df['MIN_lag_2']      = df['MIN'].shift(2)
df['RAINFALL_ROLLING_3M'] = df['RAINFALL'].rolling(window=3).mean()

df_clean = df.dropna().copy()
df_clean.index.freq = 'MS'

# 3. SYSTEM MATRICES & TRAIN-TEST SPLITTING
endog = df_clean['DENGUE']
endog_log = np.log1p(endog)

EXOG_COLS = ['MIN', 'MAX', 'HUMIDITY', 'RAINFALL',
             'HUMIDITY_lag_1', 'MIN_lag_2', 'RAINFALL_ROLLING_3M']
exog = df_clean[EXOG_COLS]

TRAIN_END  = '2024-12-01'
TEST_START = '2025-01-01'
TEST_END   = '2025-10-01'

train_endog = endog_log[:TRAIN_END]
train_exog  = exog[:TRAIN_END]
pred_dates  = exog[TEST_START:TEST_END].index

# 4. AUTOMATED HYPERPARAMETER AIC GRID SEARCH — NO SEASONAL COMPONENT
def optimize_arimax(endog_train, exog_train):
    best_aic = float("inf")
    best_order = None
    print("Executing Optimization Grid Search Across (p,d,q) Space [p:0-13, d:0-1, q:0-2]...")
    for p in range(0, 14):
        for d in range(0, 2):
            for q in range(0, 3):
                try:
                    mod = sm.tsa.statespace.SARIMAX(
                        endog=endog_train, exog=exog_train,
                        order=(p, d, q), seasonal_order=(0, 0, 0, 0),
                        enforce_stationarity=False, enforce_invertibility=False
                    )
                    res = mod.fit(disp=False, maxiter=400)
                    if res.aic < best_aic:
                        best_aic = res.aic
                        best_order = (p, d, q)
                except Exception:
                    continue
    print(f"Optimal ARIMAX Order Selected: {best_order} | Lowest AIC: {best_aic:.2f}")
    return best_order

optimal_order = optimize_arimax(train_endog, train_exog)

# 5. MODEL TRAINING & RESIDUAL DIAGNOSTICS
model = sm.tsa.statespace.SARIMAX(
    endog=train_endog, exog=train_exog,
    order=optimal_order, seasonal_order=(0, 0, 0, 0),
    enforce_stationarity=False, enforce_invertibility=False
)
results = model.fit(disp=False, maxiter=500, method='lbfgs')

# Confirm no leftover seasonal autocorrelation despite having no seasonal terms
lb = acorr_ljungbox(results.resid, lags=[6, 12, 24], return_df=True)
print("\nLjung-Box test on residuals (checks for leftover autocorrelation):")
print(lb)

fig_diag = results.plot_diagnostics(figsize=(12, 8))
fig_diag.savefig(DIAG_PNG, dpi=200)
print(f"Residual diagnostic graphics saved → {DIAG_PNG}")

# 6. WALK-FORWARD FORECASTING & ACCURACY VERIFICATION (Jan–Oct 2025)
y_pred_list, lb_list, ub_list = [], [], []

print("\nRunning walk-forward (expanding window) 1-step-ahead forecasts...")
for cutoff_date in pred_dates:
    cur_train_endog = endog_log[:cutoff_date].iloc[:-1]
    cur_train_exog  = exog.loc[cur_train_endog.index]
    step_exog       = exog.loc[[cutoff_date]]

    step_model = sm.tsa.statespace.SARIMAX(
        endog=cur_train_endog, exog=cur_train_exog,
        order=optimal_order, seasonal_order=(0, 0, 0, 0),
        enforce_stationarity=False, enforce_invertibility=False
    )
    step_res = step_model.fit(disp=False, maxiter=500, method='lbfgs')
    fc = step_res.get_forecast(steps=1, exog=step_exog)

    y_pred_list.append(np.clip(np.expm1(fc.predicted_mean.iloc[0]), 0, None))
    ci = fc.conf_int().iloc[0]
    lb_list.append(np.clip(np.expm1(ci.iloc[0]), 0, None))
    ub_list.append(np.expm1(ci.iloc[1]))
    print(f"  {cutoff_date.strftime('%b-%Y')} forecast complete")

y_true = endog[TEST_START:TEST_END].values.astype(float)
y_pred = np.array(y_pred_list)
lb_arr = np.array(lb_list)
ub_arr = np.array(ub_list)

mae  = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2   = r2_score(y_true, y_pred)

abs_pct_err  = np.where(y_true == 0, np.nan, np.abs(y_true - y_pred) / y_true * 100)
pct_accuracy = 100 - abs_pct_err
mape = np.nanmean(abs_pct_err)
overall_accuracy_pct = 100 - mape

print("\n=== 2025 (JAN–OCT) ARIMAX WALK-FORWARD PREDICTION PERFORMANCE ===")
print(f"  Order : ARIMAX{optimal_order} (no seasonal component)")
print(f"  MAE  : {mae:>10,.0f} cases")
print(f"  RMSE : {rmse:>10,.0f} cases")
print(f"  R²   : {r2:>10.4f}")
print(f"  MAPE : {mape:>10.2f} %")
print(f"  Overall Accuracy (100-MAPE) : {overall_accuracy_pct:>6.2f} %")

results_table = pd.DataFrame({
    'Month': [d.strftime('%b-%Y') for d in pred_dates],
    'Observed': y_true.astype(int),
    'Predicted': np.round(y_pred).astype(int),
    'Lower_95CI': np.round(lb_arr).astype(int),
    'Upper_95CI': np.round(ub_arr).astype(int),
    'Abs_Error': np.round(np.abs(y_true - y_pred)).astype(int),
    'Pct_Error_%': np.round(abs_pct_err, 2),
    'Pct_Accuracy_%': np.round(pct_accuracy, 2),
})
results_table.to_csv(RESULTS_CSV, index=False)
print(f"\nPer-month results saved → {RESULTS_CSV}")
print(results_table.to_string(index=False))

# 7. VISUAL DASHBOARD GENERATION
BG, PANEL, GRID      = '#0d1117', '#161b22', '#21262d'
BORDER, WHITE, MUTED = '#30363d', '#e6edf3', '#8b949e'
BLUE, ORANGE, GREEN, YELLOW = '#58a6ff', '#f78166', '#3fb950', '#d29922'

def style_ax(ax, title, ylabel='Dengue Cases'):
    ax.set_facecolor(PANEL)
    ax.set_title(title, color=WHITE, fontsize=11, fontweight='bold', pad=8)
    ax.tick_params(colors=MUTED, labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)
    ax.spines[['bottom', 'left']].set_color(BORDER)
    ax.grid(True, color=GRID, linestyle='--', linewidth=0.6, alpha=0.9)
    ax.set_ylabel(ylabel, color=MUTED, fontsize=9)

fig = plt.figure(figsize=(15, 14), dpi=150)
fig.patch.set_facecolor(BG)
gs  = fig.add_gridspec(4, 2, hspace=0.55, wspace=0.32)
ax1 = fig.add_subplot(gs[0, :])
ax2 = fig.add_subplot(gs[1, :])
ax3 = fig.add_subplot(gs[2, 0])
ax4 = fig.add_subplot(gs[2, 1])
ax5 = fig.add_subplot(gs[3, :])

style_ax(ax1, '▸ Walk-Forward System View: Dengue Timeline Verification (2008 – 2025)')
ax1.plot(train_endog.index, np.expm1(train_endog.values), color=MUTED, lw=1.3, alpha=0.55, label='Historical Records')
ax1.plot(pred_dates, y_true, color=BLUE, lw=2, marker='o', ms=4, label='Observed 2025')
ax1.plot(pred_dates, y_pred, color=ORANGE, lw=2, ls='--', marker='x', ms=5, label='ARIMAX Forecast')
ax1.fill_between(pred_dates, lb_arr, ub_arr, color=ORANGE, alpha=0.15, label='95% CI Boundary')
ax1.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER, loc='upper left', ncol=3)

style_ax(ax2, '▸ 2025 (Jan–Oct) Walk-Forward Accuracy Resolution')
ax2.plot(pred_dates, y_true, color=BLUE, lw=2.3, marker='o', ms=6, label='Observed Cases (DGHS)', zorder=4)
ax2.plot(pred_dates, y_pred, color=ORANGE, lw=2.3, ls='--', marker='x', ms=7, label='ARIMAX Prediction', zorder=4)
ax2.fill_between(pred_dates, lb_arr, ub_arr, color=ORANGE, alpha=0.18, label='95% Predictive Envelope')

obs_pk, pred_pk = np.argmax(y_true), np.argmax(y_pred)
ax2.annotate(f'Observed Peak\n{int(y_true[obs_pk]):,}', xy=(pred_dates[obs_pk], y_true[obs_pk]),
             xytext=(pred_dates[max(obs_pk-2,0)], y_true[obs_pk]*0.80), color=YELLOW, fontsize=9, arrowprops=dict(arrowstyle='->', color=YELLOW, lw=1.2))
ax2.annotate(f'Predicted Peak\n{int(y_pred[pred_pk]):,}', xy=(pred_dates[pred_pk], y_pred[pred_pk]),
             xytext=(pred_dates[max(pred_pk-2,0)], y_pred[pred_pk]*1.15), color=ORANGE, fontsize=9, arrowprops=dict(arrowstyle='->', color=ORANGE, lw=1.2))

ax2.text(0.99, 0.04, f'MAE: {mae:,.0f} | RMSE: {rmse:,.0f} | R²: {r2:.4f} | Accuracy: {overall_accuracy_pct:.1f}%',
         transform=ax2.transAxes, ha='right', va='bottom', fontsize=9.5, color=WHITE,
         bbox=dict(boxstyle='round,pad=0.5', facecolor=BG, edgecolor=BORDER, alpha=0.92))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=1))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax2.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER, loc='upper left')

style_ax(ax3, '▸ Monthly Convergence Distribution')
months_lbl = [d.strftime('%b') for d in pred_dates]
x, w = np.arange(len(pred_dates)), 0.38
ax3.bar(x - w/2, y_true, w, color=BLUE, alpha=0.85, label='Observed', zorder=3)
ax3.bar(x + w/2, y_pred, w, color=ORANGE, alpha=0.85, label='Predicted', zorder=3)
ax3.set_xticks(x)
ax3.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax3.legend(fontsize=8.5, facecolor=PANEL, labelcolor=WHITE, edgecolor=BORDER)

style_ax(ax4, '▸ Predictive Error Volatility Analysis', ylabel='Residual Cases')
residuals = y_true - y_pred
res_colors = [GREEN if r >= 0 else ORANGE for r in residuals]
ax4.bar(range(len(pred_dates)), residuals, color=res_colors, alpha=0.85, zorder=3)
ax4.axhline(0, color=WHITE, lw=0.9, ls='--', alpha=0.5)
ax4.set_xticks(range(len(pred_dates)))
ax4.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)

style_ax(ax5, '▸ Per-Month Prediction Accuracy (%)', ylabel='Accuracy (%)')
bar_colors = [GREEN if a >= 80 else (YELLOW if a >= 60 else ORANGE) for a in np.nan_to_num(pct_accuracy, nan=0)]
ax5.bar(range(len(pred_dates)), np.nan_to_num(pct_accuracy, nan=0), color=bar_colors, alpha=0.9, zorder=3)
ax5.axhline(100, color=WHITE, lw=0.9, ls='--', alpha=0.4)
ax5.set_xticks(range(len(pred_dates)))
ax5.set_xticklabels(months_lbl, fontsize=8.5, color=MUTED)
ax5.set_ylim(min(0, np.nanmin(pct_accuracy) - 5), 105)
for i, a in enumerate(pct_accuracy):
    label = f"{a:.0f}%" if not np.isnan(a) else "N/A"
    ax5.text(i, (0 if np.isnan(a) else a) + (2 if (not np.isnan(a) and a >= 0) else -8), label,
              ha='center', fontsize=8, color=WHITE)

fig.suptitle(f'Walk-Forward ARIMAX Early Warning Matrix — Bangladesh 2025 (Jan–Oct)\nSelected Order: ARIMAX{optimal_order} (no seasonal component) | Exog: {len(EXOG_COLS)} (VIF-screened) | Accuracy: {overall_accuracy_pct:.1f}%',
             color=WHITE, fontsize=12, fontweight='bold', y=0.999)

plt.savefig(OUTPUT_PNG, bbox_inches='tight', facecolor=BG, dpi=150)
print(f"Publication-ready verification matrix saved → {OUTPUT_PNG}")

Executing Optimization Grid Search Across (p,d,q) Space [p:0-13, d:0-1, q:0-2]...
Optimal ARIMAX Order Selected: (13, 0, 0) | Lowest AIC: 516.68

Ljung-Box test on residuals (checks for leftover autocorrelation):
      lb_stat  lb_pvalue
6    9.935907   0.127377
12  12.866640   0.378808
24  20.817427   0.649471
Residual diagnostic graphics saved → arimax_residual_diagnostics.png

Running walk-forward (expanding window) 1-step-ahead forecasts...
  Jan-2025 forecast complete
  Feb-2025 forecast complete
  Mar-2025 forecast complete
  Apr-2025 forecast complete
  May-2025 forecast complete
  Jun-2025 forecast complete
  Jul-2025 forecast complete
  Aug-2025 forecast complete
  Sep-2025 forecast complete
  Oct-2025 forecast complete

=== 2025 (JAN–OCT) ARIMAX WALK-FORWARD PREDICTION PERFORMANCE ===
  Order : ARIMAX(13, 0, 0) (no seasonal component)
  MAE  :      1,370 cases
  RMSE :      2,157 cases
  R²   :     0.8767
  MAPE :      21.36 %
  Overall Accuracy (100-MAPE) :  78.64 %

Per-mon